# PREPROCESSING PIPELINE

In [3]:
import numpy as np
import pandas as pd
import pyreadr

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit

## 1. LOAD RDS FILE

In [4]:
RDS_PATH = "data/features_all_windows_combined.rds"

result = pyreadr.read_r(RDS_PATH)
df = next(iter(result.values()))
df = pd.DataFrame(df)

## 2. REMOVE ACCIDENTAL INDEX / UNNAMED COLUMNS

In [5]:
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]

## 3. TIME FEATURE ENGINEERING

In [6]:
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

df["session_duration_sec"] = (
    df["end_time"] - df["start_time"]
).dt.total_seconds().clip(lower=1)

hour = df["start_time"].dt.hour
df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24)

dow = df["start_time"].dt.weekday
df["dow_sin"] = np.sin(2 * np.pi * dow / 7)
df["dow_cos"] = np.cos(2 * np.pi * dow / 7)

df = df.drop(columns=["start_time", "end_time"])

## 4. BINARY FEATURES (KEEP AS 0/1)

In [7]:
binary_cols = [
    "gps_home", "gps_work",
    "headset_PLUGGED", "headset_UNPLUGGED",
    "phone_AIRPLANE", "phone_BATTERYSAVINGMODE",
    "phone_BLUETOOTH", "phone_CAMERA", "phone_CALENDAR",
    "phone_PHONE", "phone_POWER", "phone_SMS"
]

binary_cols = [c for c in binary_cols if c in df.columns]

for col in binary_cols:
    df[col] = (df[col] > 0).astype(int)

## 5. MUSIC KEY AS CATEGORICAL


In [8]:
if "music_track_key" in df.columns:
    df["music_track_key"] = (
        df["music_track_key"]
        .round()
        .astype("Int64")
        .astype("category")
    )

## 6. COLLAPSE RARE CATEGORICAL LEVELS

In [9]:
def collapse_rare(series, min_freq=0.0001):
    freq = series.value_counts(normalize=True)
    rare = freq[freq < min_freq].index
    return series.where(~series.isin(rare), other="Other")

for col in ["gps_landuse_type", "time_bin"]:
    if col in df.columns:
        df[col] = collapse_rare(df[col])


## 7. DEFINE FEATURE GROUPS


In [10]:
# user_id stays as-is
pass_through_features = ["user_id"]

categorical_features = [
    "time_bin",
    "gps_landuse_type",
    "music_track_key"
]
categorical_features = [c for c in categorical_features if c in df.columns]

# numeric features exclude user_id and binary columns
numeric_features = [
    c for c in df.columns
    if c not in categorical_features + binary_cols + pass_through_features
]

## 8. PREPROCESSING PIPELINES

In [11]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

binary_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("bin", binary_pipeline, binary_cols),
        ("cat", categorical_pipeline, categorical_features),
        ("pass", "passthrough", pass_through_features)  # <- user_id untouched
    ]
)

## 9. TRAIN/TEST SPLIT

In [12]:
groups = df["user_id"].values
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42  # Fixed for reproducibility
)

train_idx, test_idx = next(gss.split(df, groups=groups))

# Split data BEFORE preprocessing to avoid leakage
df_train = df.iloc[train_idx]
df_test = df.iloc[test_idx]

# Fit preprocessor ONLY on training data
preprocessor.fit(df_train)

# Transform both train and test
train = preprocessor.transform(df_train)
test = preprocessor.transform(df_test)

# Get feature names
feature_names = preprocessor.get_feature_names_out()
train = pd.DataFrame(train, columns=feature_names)
test = pd.DataFrame(test, columns=feature_names)

output_prefixes = (
    "num__liwc_",
    "num__topic_",
    "num__genius_",
    "num__music_track_",
)
output_cols = [c for c in df.columns if c.startswith(output_prefixes)]

X_train = train.drop(columns=output_cols)
y_train = train[output_cols]

X_test = test.drop(columns=output_cols)
y_test = test[output_cols]

/home/cjs/miniconda3/envs/idp/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/cjs/miniconda3/envs/idp/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:83: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/cjs/miniconda3/envs/idp/lib/python3.11/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['liwc_Sixltr' 'liwc_function.' 'liwc_you_total' 'liwc_you_sing'
 'liwc_you_plur' 'liwc_you_formal' 'liwc_other' 'liwc_compare'
 'liwc_interrog' 'liwc_quant' 'liwc_affect' 'liwc_posemo' 'liwc_negemo'
 'liwc_anx' 'liwc_sad' 'liwc_social' 'liwc_certain' 'liwc_percept'
 'liwc_see' 'liwc_hear' 'liwc_feel' 'liwc_bio' 'liwc_body' 'liwc_ingest'
 'liwc_drives' 'liwc_achiev' 'liwc_relativ' 'liwc_informal' 'liwc_Colon'
 'liwc_SemiC' 'liwc_Dash' 'liwc_Quote

## 10. SAVE PREPROCESSED DATA

In [13]:
OUTPUT_DIR = "data/preprocessed"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save features
X_train.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)

# Save targets
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

print("Preprocessing complete. Files saved to:")
print(f"- Features: {OUTPUT_DIR}/X_train.csv, {OUTPUT_DIR}/X_test.csv")
print(f"- Targets: {OUTPUT_DIR}/y_train.csv, {OUTPUT_DIR}/y_test.csv")

Preprocessing complete. Files saved to:
- Features: data/preprocessed/X_train.csv, data/preprocessed/X_test.csv
- Targets: data/preprocessed/y_train.csv, data/preprocessed/y_test.csv
